# Multi-Condition Risk Models
- Reused the heart attack model approach from Models.ipynb to predict every condition in the dataset heart attack, angina, stroke, asthma, skin cancer, COPD, depression, kidney disease, arthritis, and diabetes
- Same setup as the heart attack models, stratified 80/20 split, stratified 5-fold CV, balanced class weights, PR-AUC to pick the model, and a threshold that catches 80% of positive cases
- Same two feature sets. full (all other columns, including the other conditions) and lifestyle (no condition columns, so the app can work from things a user knows about themselves)
- Each final model is saved as one sklearn Pipeline (preprocessing + classifier) in models, so the app can pass in raw values and get a probability back
- Heart attack is run through the same loop as a sanity check, its CV scores should match Models.ipynb

In [1]:
import os
import json
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, fbeta_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.isotonic import IsotonicRegression

# MLflow is optional so the notebook still runs on machines without it
try:
    import mlflow
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("condition_models")
    USE_MLFLOW = True
except ImportError:
    USE_MLFLOW = False
print("MLflow logging:", USE_MLFLOW)

2026/10/07 10:28:10 INFO mlflow.tracking.fluent: Experiment with name 'condition_models' does not exist. Creating a new experiment.


MLflow logging: True


### Load Data
- HadDiabetes is now converted to 0/1
- Only "Yes" counts as diabetes. pre-diabetes and diabetes only during pregnancy count as "No" (matches the 13.7% rate in Descriptive_Stats.ipynb)
- State is dropped, same as in Modeling.ipynb
- HadDepressiveDisorder is renamed to HadDepression to match the project naming

In [2]:
df = pd.read_csv("heart_2022_PREPROCESSED.csv")

DIABETES_MAPPING = {
    "Yes": 1,
    "No": 0,
    "No, pre-diabetes or borderline diabetes": 0,
    "Yes, but only during pregnancy (female)": 0,
}
df["HadDiabetes"] = df["HadDiabetes"].map(DIABETES_MAPPING)
df = df.drop(columns=["State"])
df = df.rename(columns={"HadDepressiveDisorder": "HadDepression"})

print(df.shape)
print("Missing values:", df.isnull().sum().sum())

(246013, 39)
Missing values: 0


### Columns and Feature Sets
- Same numeric/categorical column lists as Modeling.ipynb, plus HadHeartAttack and HadDiabetes, which are now features for the other models
- Each model drops its own target from the inputs
- Lifestyle drops every condition column and ChestScan, same as Models.ipynb

In [3]:
TARGETS = [
    "HadHeartAttack",
    "HadAngina",
    "HadStroke",
    "HadAsthma",
    "HadSkinCancer",
    "HadCOPD",
    "HadDepression",
    "HadKidneyDisease",
    "HadArthritis",
    "HadDiabetes",
]

CATEGORICAL_COLS = ["Sex", "ECigaretteUsage", "RaceEthnicityCategory", "TetanusLast10Tdap", "CovidPos"]

NUMERIC_COLS = [
    "GeneralHealth", "AgeCategory", "SmokerStatus", "LastCheckupTime", "RemovedTeeth",
    "PhysicalHealthDays", "MentalHealthDays", "SleepHours",
    "HeightInMeters", "WeightInKilograms", "BMI",
    "PhysicalActivities", "DeafOrHardOfHearing", "BlindOrVisionDifficulty",
    "DifficultyConcentrating", "DifficultyWalking", "DifficultyDressingBathing",
    "DifficultyErrands", "ChestScan", "AlcoholDrinkers", "HIVTesting",
    "FluVaxLast12", "PneumoVaxEver", "HighRiskLastYear",
] + TARGETS  # every condition is a numeric 0/1 input for the other models

CONDITION_COLS = TARGETS + ["ChestScan"]

def feature_columns(target, feature_set):
    """Returns (numeric, categorical) input columns for one target and feature set."""
    numeric = [c for c in NUMERIC_COLS if c != target]
    if feature_set == "lifestyle":
        numeric = [c for c in numeric if c not in CONDITION_COLS]
    return numeric, CATEGORICAL_COLS

for fs in ["full", "lifestyle"]:
    num, cat = feature_columns("HadAngina", fs)
    print(f"{fs}: {len(num)} numeric + {len(cat)} categorical raw columns")

full: 33 numeric + 5 categorical raw columns
lifestyle: 23 numeric + 5 categorical raw columns


### Pipeline Builder
- Same preprocessing as Modeling.ipynb (median impute + scale numeric, most-frequent impute + one-hot categorical), but bundled with the model
- Fitting the pipeline inside CV means the scaler is refit on each training fold, so nothing leaks from the validation fold

In [4]:
def make_preprocessor(numeric, categorical):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical),
    ])

# Tuned settings from Models.ipynb (RandomizedSearchCV on HadHeartAttack), reused for every condition
MODELS = {
    "logreg": lambda fs: LogisticRegression(class_weight="balanced", max_iter=1000, C=0.002),
    "hgb": lambda fs: HistGradientBoostingClassifier(
        class_weight="balanced", max_iter=500, random_state=42,
        **({"learning_rate": 0.092, "max_leaf_nodes": 23, "min_samples_leaf": 109} if fs == "full"
           else {"learning_rate": 0.056, "max_leaf_nodes": 17, "min_samples_leaf": 70}),
    ),
}

def make_pipeline(target, feature_set, model_name):
    numeric, categorical = feature_columns(target, feature_set)
    return Pipeline([
        ("preprocess", make_preprocessor(numeric, categorical)),
        ("model", MODELS[model_name](feature_set)),
    ])

### Evaluation
- One cross_val_predict per model gives out-of-fold probabilities for every training row, and every metric is computed from those
- Metrics match Models.ipynb, PR-AUC, ROC-AUC, and recall/precision/F2 at the 0.5 cutoff
- thr_80 is the highest threshold that still catches 80% of cases, and prec_at_80 is the precision at that threshold

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
TARGET_RECALL = 0.80

def threshold_for_recall(y_true, proba, target_recall=TARGET_RECALL):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    return float(thresholds[recall[:-1] >= target_recall].max())

def score(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "recall": recall_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f2": fbeta_score(y_true, pred, beta=2),
    }

### Train/Test Split per Condition
- Same 80/20 split with random_state=42, stratified on each condition so the train and test rates match
- For HadHeartAttack this is the same split as Modeling.ipynb

In [6]:
splits = {}
for target in TARGETS:
    X = df.drop(columns=[target])
    y = df[target]
    splits[target] = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

pd.DataFrame({
    "positive_rate_%": [round(df[t].mean() * 100, 2) for t in TARGETS],
    "train_positives": [int(splits[t][2].sum()) for t in TARGETS],
}, index=TARGETS)

,positive_rate_%,train_positives
HadHeartAttack,5.46,10748
HadAngina,6.08,11962
HadStroke,4.11,8089
HadAsthma,14.85,29222
HadSkinCancer,8.54,16815
HadCOPD,7.72,15195
HadDepression,20.58,40496
HadKidneyDisease,4.59,9027
HadArthritis,34.50,67905
HadDiabetes,13.74,27049


### Cross-Validate Every Condition
10 conditions, 2 feature sets, 2 models, each with 5-fold CV

In [7]:
cv_results = []
oof = {}  # out-of-fold probabilities, reused for thresholds

start = time.time()
for target in TARGETS:
    X_train, X_test, y_train, y_test = splits[target]
    for fs in ["full", "lifestyle"]:
        for model_name in MODELS:
            pipe = make_pipeline(target, fs, model_name)
            proba = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
            oof[(target, fs, model_name)] = proba

            row = {"target": target, "feature_set": fs, "model": model_name}
            row.update(score(y_train, proba))
            thr = threshold_for_recall(y_train, proba)
            row["thr_80"] = thr
            row["prec_at_80"] = precision_score(y_train, (proba >= thr).astype(int))
            cv_results.append(row)

            if USE_MLFLOW:
                with mlflow.start_run(run_name=f"{target}_{model_name}_{fs}"):
                    mlflow.log_params({"target": target, "model": model_name, "feature_set": fs})
                    mlflow.log_metrics({k: v for k, v in row.items() if isinstance(v, float)})
    print(f"{target} done ({time.time() - start:.0f}s elapsed)")

cv_df = pd.DataFrame(cv_results)
cv_df.round(3)

HadHeartAttack done (101s elapsed)
HadAngina done (172s elapsed)
HadStroke done (240s elapsed)
HadAsthma done (321s elapsed)
HadSkinCancer done (389s elapsed)
HadCOPD done (466s elapsed)
HadDepression done (558s elapsed)
HadKidneyDisease done (626s elapsed)
HadArthritis done (700s elapsed)
HadDiabetes done (778s elapsed)


,target,feature_set,model,pr_auc,roc_auc,recall,precision,f2,thr_80,prec_at_80
0,HadHeartAttack,full,logreg,0.407,0.890,0.768,0.214,0.506,0.457,0.193
1,HadHeartAttack,full,hgb,0.416,0.890,0.791,0.201,0.499,0.486,0.195
2,HadHeartAttack,lifestyle,logreg,0.215,0.826,0.777,0.142,0.410,0.478,0.136
3,HadHeartAttack,lifestyle,hgb,0.219,0.828,0.795,0.139,0.409,0.495,0.138
4,HadAngina,full,logreg,0.439,0.895,0.800,0.225,0.530,0.500,0.225
5,HadAngina,full,hgb,0.440,0.896,0.816,0.217,0.526,0.520,0.225
6,HadAngina,lifestyle,logreg,0.247,0.839,0.798,0.161,0.446,0.497,0.160
7,HadAngina,lifestyle,hgb,0.249,0.840,0.815,0.157,0.443,0.517,0.161
8,HadStroke,full,logreg,0.182,0.825,0.746,0.114,0.353,0.443,0.101
9,HadStroke,full,hgb,0.185,0.826,0.778,0.108,0.347,0.473,0.103


### Sanity Check Against Models.ipynb
Heart attack CV scores should be close to the tuned results (full: PR-AUC ~0.405 logreg / ~0.415 hgb, lifestyle: ~0.216 / ~0.219)

In [8]:
cv_df[cv_df["target"] == "HadHeartAttack"].round(3)

,target,feature_set,model,pr_auc,roc_auc,recall,precision,f2,thr_80,prec_at_80
0,HadHeartAttack,full,logreg,0.407,0.890,0.768,0.214,0.506,0.457,0.193
1,HadHeartAttack,full,hgb,0.416,0.890,0.791,0.201,0.499,0.486,0.195
2,HadHeartAttack,lifestyle,logreg,0.215,0.826,0.777,0.142,0.410,0.478,0.136
3,HadHeartAttack,lifestyle,hgb,0.219,0.828,0.795,0.139,0.409,0.495,0.138


### Pick the Best Model per Condition
- Best = highest CV PR-AUC for each condition and feature set
- The final model is refit on the whole training set, and its 80% recall threshold comes from the CV probabilities (not the test set)

In [9]:
best = (cv_df.sort_values("pr_auc", ascending=False)
             .groupby(["target", "feature_set"], sort=False).head(1)
             .set_index(["target", "feature_set"])
             .loc[[(t, fs) for t in TARGETS for fs in ["full", "lifestyle"]]])
best[["model", "pr_auc", "roc_auc", "thr_80", "prec_at_80"]].round(3)

model  pr_auc  roc_auc  thr_80  prec_at_80
target           feature_set                                           
HadHeartAttack   full          hgb   0.416    0.890   0.486       0.195
                 lifestyle     hgb   0.219    0.828   0.495       0.138
HadAngina        full          hgb   0.440    0.896   0.520       0.225
                 lifestyle     hgb   0.249    0.840   0.517       0.161
HadStroke        full          hgb   0.185    0.826   0.473       0.103
                 lifestyle     hgb   0.151    0.805   0.462       0.093
HadAsthma        full          hgb   0.345    0.718   0.386       0.210
                 lifestyle     hgb   0.295    0.687   0.400       0.197
HadSkinCancer    full          hgb   0.249    0.802   0.516       0.181
                 lifestyle     hgb   0.245    0.800   0.520       0.180
HadCOPD          full          hgb   0.452    0.879   0.505       0.251
                 lifestyle     hgb   0.390    0.854   0.462       0.211
HadDepression    full          hgb   0.627    0.841   0.429       0.426
                 lifestyle     hgb   0.621    0.838   0.426       0.422
HadKidneyDisease full          hgb   0.191    0.816   0.457       0.108
                 lifestyle     hgb   0.166    0.800   0.459       0.102
HadArthritis     full          hgb   0.685    0.814   0.473       0.561
                 lifestyle     hgb   0.679    0.811   0.473       0.555
HadDiabetes      full          hgb   0.432    0.831   0.496       0.304
                 lifestyle     hgb   0.416    0.828   0.500       0.301

### Refit, Test, and Save
- Each final pipeline is scored once on the held out 20% test set at its chosen threshold
- Saved to models/{target}_{feature_set}.joblib as a dict with the pipeline, threshold, input columns, and test metrics
- models/manifest.json lists every saved model so the app will be able to load them all at startup

In [10]:
os.makedirs("models", exist_ok=True)
test_results = []
manifest = {}
test_preds = {}

for (target, fs), row in best.iterrows():
    X_train, X_test, y_train, y_test = splits[target]
    pipe = make_pipeline(target, fs, row["model"]).fit(X_train, y_train)
    threshold = row["thr_80"]

    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    test_preds[(target, fs)] = pred

    metrics = score(y_test, proba)
    metrics["recall_at_thr"] = recall_score(y_test, pred)
    metrics["precision_at_thr"] = precision_score(y_test, pred)
    metrics["base_rate"] = float(y_test.mean())

    numeric, categorical = feature_columns(target, fs)
    bundle = {
        "pipeline": pipe,
        "target": target,
        "feature_set": fs,
        "model": row["model"],
        "threshold": threshold,
        "input_columns": numeric + categorical,
        "test_metrics": metrics,
    }
    bundle["calibrator"] = IsotonicRegression(out_of_bounds="clip").fit(oof[(target, fs, row["model"])], y_train)
    path = f"models/{target}_{fs}.joblib"
    joblib.dump(bundle, path)
    manifest[f"{target}_{fs}"] = {k: v for k, v in bundle.items() if k not in ("pipeline", "calibrator")} | {"path": path}

    test_results.append({"target": target, "feature_set": fs, "model": row["model"], "threshold": threshold, **metrics})

with open("models/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

test_df = pd.DataFrame(test_results)
test_df[["target", "feature_set", "model", "base_rate", "pr_auc", "roc_auc",
         "threshold", "recall_at_thr", "precision_at_thr"]].round(3)

,target,feature_set,model,base_rate,pr_auc,roc_auc,threshold,recall_at_thr,precision_at_thr
0,HadHeartAttack,full,hgb,0.055,0.413,0.891,0.486,0.806,0.196
1,HadHeartAttack,lifestyle,hgb,0.055,0.213,0.826,0.495,0.791,0.135
2,HadAngina,full,hgb,0.061,0.457,0.901,0.520,0.806,0.224
3,HadAngina,lifestyle,hgb,0.061,0.252,0.841,0.517,0.796,0.159
4,HadStroke,full,hgb,0.041,0.185,0.825,0.473,0.792,0.101
5,HadStroke,lifestyle,hgb,0.041,0.146,0.806,0.462,0.802,0.093
6,HadAsthma,full,hgb,0.148,0.360,0.729,0.386,0.814,0.214
7,HadAsthma,lifestyle,hgb,0.148,0.308,0.701,0.400,0.816,0.200
8,HadSkinCancer,full,hgb,0.085,0.249,0.803,0.516,0.813,0.182
9,HadSkinCancer,lifestyle,hgb,0.085,0.248,0.801,0.520,0.807,0.181


### Load Check
Reload one saved model and score a single raw row, the same way the app will

In [11]:
bundle = joblib.load("models/HadDiabetes_lifestyle.joblib")
example = splits["HadDiabetes"][1].iloc[[0]][bundle["input_columns"]]
p = bundle["pipeline"].predict_proba(example)[0, 1]
print("Inputs the model expects:", len(bundle["input_columns"]))
print(f"Predicted diabetes probability: {p:.3f}  (flagged: {p >= bundle['threshold']})")

Inputs the model expects: 28
Predicted diabetes probability: 0.787  (flagged: True)


In [ ]:
# Check calibration
b = joblib.load("models/HadHeartAttack_lifestyle.joblib")
p = b["pipeline"].predict_proba(splits["HadHeartAttack"][1])[:, 1]
print("before:", p.mean().round(3), "after:", b["calibrator"].predict(p).mean().round(3))

before: 0.35 after: 0.055
